<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_100_fairness_bias_explainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⚖️ **Module 100 — Fairness · Bias Mitigation · Explainability** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 14 · Theory & Responsibility · **FINAL MODULE** (100/100)


# Module 100 — Fairness · Bias Mitigation · Explainability

> The 100th and final module. UDL **Chapter 21** ships two notebooks (**Bias Mitigation · Explainability**) and the rest of the course brushed past these topics — they're the **regulatory pillar** of every shipping ML system in 2026. EU AI Act (Aug 2024), Colorado SB-205, NYC AEDT, California SB-942, and FDA SaMD all require some combination of **bias evaluation · post-hoc explanation · auditability · user appeal**. This module is the complete recipe: the **fairness metric zoo** (and which one is *legally required* where), **pre/in/post-training bias mitigation**, the **explainability canon** (SHAP, LIME, IG, GradCAM, attention attribution, Captum), **counterfactual** + **contrastive** explanations, and the **2026 production stack** for shipping responsible ML.

### What you'll cover
1. The **legal frame** — EU AI Act, Colorado, NYC, FDA, Brazil LGPD
2. The **fairness metric zoo** — demographic parity · equal opportunity · equalized odds · calibration parity
3. The **impossibility theorem** (Kleinberg, Chouldechova) — why you can't satisfy all metrics simultaneously
4. **Pre-processing** bias mitigation — reweighing, sampling, learning-fair-representations
5. **In-processing** — adversarial debiasing, fair-learn constraints, Reductions
6. **Post-processing** — equalized-odds / calibration post-hoc fix (Hardt 2016)
7. **SHAP · LIME · Integrated Gradients · GradCAM** — the explainability canon
8. **Counterfactual explanations** + **anchors** + **contrastive**
9. **LLM-specific responsibility** — Constitutional AI (M89) · attention attribution · jailbreak evals (M91, M95)
10. The **2026 production stack** + the 100-module epilogue


## 1 · The legal frame

Anywhere a model affects people, regulation now applies. The 2026 picture:

| Jurisdiction | Law / Rule | What it requires |
|---|---|---|
| **EU** | **AI Act** (Aug 2024, fully effective Aug 2026) | Risk classification; "high-risk" systems need conformity assessment, bias evaluation, **technical documentation**, human oversight, transparency to affected persons |
| **Colorado** | **SB-205** (Feb 2026 effective) | Annual impact assessments for "consequential decisions" — employment, housing, credit, insurance, healthcare, education |
| **NYC** | **AEDT Law** (2023) | Annual bias audits + public reporting for automated employment decision tools |
| **California** | **SB-942** + GenAI watermarking | Watermarking for AI-generated content; transparency duties |
| **US Federal** | **Algorithmic Accountability Act** (proposed), FTC Section 5 | Discriminatory pricing / hiring; consumer-protection enforcement |
| **FDA (US)** | **SaMD framework + AI/ML Action Plan** | Medical AI: predetermined-change-control plan, post-market monitoring |
| **Brazil** | **LGPD + ANPD AI Resolutions** | Right to explanation, opt-out of automated decisions |
| **China** | **Algorithm Recommendation Regulations** + **Generative AI Measures** | Registration, watermarking, content filtering |
| **OECD AI Principles** | Soft-law framework | Drives convergence of the above |

**The two recurring requirements:** bias audit (statistical fairness metrics) + post-hoc explanation (why was *this* user denied?). Skipping either now creates corporate liability, not just reputational risk.

> 📋 **Practical hook.** A 2026 model-cards or system-card document (Mitchell 2019 → Anthropic system cards 2024) typically includes: intended use, training-data summary, **subgroup performance breakdown**, **fairness metric report**, known limitations, appeals process. Every frontier lab now publishes one with each release.


## 2 · The fairness metric zoo

Each metric corresponds to a different *intuition* about what "fair" means. Pick one before training — you cannot satisfy all of them simultaneously (Section 3).

Notation: `A` = protected attribute (race, sex, age-class), `Y` = true label, `Ŷ` = prediction.

| Metric | Definition | Says |
|---|---|---|
| **Demographic Parity (DP)** | `P(Ŷ=1 | A=0) = P(Ŷ=1 | A=1)` | Selection rate equal across groups |
| **Equal Opportunity (EOpp)** (Hardt 2016) | `P(Ŷ=1 | A=0, Y=1) = P(Ŷ=1 | A=1, Y=1)` | True-positive rate equal across groups |
| **Equalized Odds (EO)** | TPR equal **and** FPR equal across groups | EOpp + symmetric FPR constraint |
| **Predictive Parity** | `P(Y=1 | Ŷ=1, A=0) = P(Y=1 | Ŷ=1, A=1)` | Precision / positive predictive value equal |
| **Calibration Parity** | `P(Y=1 | p̂=p, A=0) = P(Y=1 | p̂=p, A=1) = p` | Predicted probability means the same thing across groups |
| **Counterfactual Fairness** (Kusner 2017) | `Ŷ(x, A=a) = Ŷ(x, A=a')` for all `a, a'` | Causal — changing `A` (counterfactually) doesn't change the prediction |
| **Individual Fairness** (Dwork 2012) | Similar individuals get similar predictions; "Lipschitz" w.r.t. a fair similarity metric | Person-level, harder to operationalize |
| **Disparate Impact / 80% rule** | `min_a P(Ŷ=1|A=a) / max_a P(Ŷ=1|A=a) ≥ 0.8` | US legal precedent (EEOC) |

**Choose by use case:**
- **Hiring / lending / housing** → Demographic Parity or 80% rule (US legal)
- **Medical triage** → Equal Opportunity (don't miss positives differentially)
- **Credit scoring** → Equalized Odds + Calibration
- **Risk-scoring (recidivism, fraud)** → Calibration + EO (ProPublica vs. Northpointe COMPAS debate)


In [ ]:
import numpy as np

def fairness_metrics(y_true, y_pred, group):
    """Compute the textbook fairness metrics in one shot."""
    out = {}
    for g in np.unique(group):
        mask = group == g
        yt, yp = y_true[mask], y_pred[mask]
        out[f"sel_rate[{g}]"] = yp.mean()                                     # P(Ŷ=1)
        out[f"TPR[{g}]"]      = yp[yt == 1].mean()                            # P(Ŷ=1|Y=1)
        out[f"FPR[{g}]"]      = yp[yt == 0].mean()                            # P(Ŷ=1|Y=0)
        out[f"PPV[{g}]"]      = yt[yp == 1].mean()                            # P(Y=1|Ŷ=1)
    out["DP_gap"]   = max(out[k] for k in out if k.startswith("sel"))  \
                    - min(out[k] for k in out if k.startswith("sel"))
    out["EOpp_gap"] = max(out[k] for k in out if k.startswith("TPR"))  \
                    - min(out[k] for k in out if k.startswith("TPR"))
    out["EO_gap"]   = max(out["EOpp_gap"],
                          max(out[k] for k in out if k.startswith("FPR")) -
                          min(out[k] for k in out if k.startswith("FPR")))
    out["80pct_rule"] = (min(out[k] for k in out if k.startswith("sel"))
                        /max(out[k] for k in out if k.startswith("sel")))
    return out


## 3 · The impossibility theorem

**Chouldechova 2017** + **Kleinberg, Mullainathan, Raghavan 2017** proved a discomforting fact: when base rates differ across groups (`P(Y=1 | A=0) ≠ P(Y=1 | A=1)`), it is **mathematically impossible** to simultaneously achieve **calibration parity, equalized FPR, and equalized FNR** across groups, unless the model is *perfect*.

This is the **ProPublica vs. Northpointe COMPAS debate** in one theorem. ProPublica said COMPAS was unfair because the FPR for Black defendants was higher than for White. Northpointe responded that COMPAS was *calibrated* across groups (a 7/10 risk score means the same recidivism probability regardless of race). **Both were right.** The base rates differed; the theorem says you cannot satisfy both rubrics simultaneously.

```
Calibration  +  Equal FPR  +  Equal FNR  ⇒  perfect model OR equal base rates
```

**Implications:**
1. **Pick a fairness metric per problem and defend that choice** — there is no universal "fair model"
2. **Base rates matter as much as the model** — if `P(Y=1)` differs by group, fairness trade-offs are unavoidable
3. **Document the choice** — required by EU AI Act, recommended by NIST AI Risk Management Framework
4. **Communicate to stakeholders** — affected users are entitled to know which definition was used

> ⚖️ **The intellectual move.** This theorem turned fairness in ML from "find the magic algorithm" to a **policy choice with quantifiable trade-offs**. Modern fair-ML stacks (Fairlearn, AIF360) let you sweep the trade-off frontier and choose the operating point.


## 4 · Pre-processing — fix the data

| Method | Idea |
|---|---|
| **Reweighing** (Kamiran & Calders 2012) | Multiply each sample's weight by `P(A)·P(Y) / P(A, Y)` so `A ⊥ Y` in the weighted distribution |
| **Resampling** | Over-sample minority-positive and majority-negative; under-sample the other two cells |
| **Suppression / Disparate-Impact-Remover** (Feldman 2015) | Modify features so that `A` becomes statistically independent of them |
| **Learning Fair Representations (LFR)** (Zemel 2013) | Train an encoder that maps `x` to a representation `z` from which `A` cannot be recovered |
| **Variational Fair Autoencoder** (Louizos 2016) | LFR + reconstruction; preserves utility |
| **Data augmentation** | Counterfactual data augmentation — flip `A`, generate matched examples |

**Reweighing in 5 lines** is the textbook starting point — fast, gives ~30-60% of available bias-mitigation gains, no model changes.

```python
def reweighing_weights(y, a):
    P = {(yi, ai): ((y == yi) & (a == ai)).mean() for yi in {0,1} for ai in {0,1}}
    Py, Pa = {yi: (y == yi).mean() for yi in {0,1}}, {ai: (a == ai).mean() for ai in {0,1}}
    return [(Py[yi] * Pa[ai]) / P[(yi, ai)] for yi, ai in zip(y, a)]
```

> ✏️ **Trade-off.** Pre-processing is least intrusive (downstream stack unchanged) but loses to in-processing on accuracy at any given fairness level. Often combined with post-processing for layered safety.


## 5 · In-processing — fix the objective

| Method | Idea |
|---|---|
| **Adversarial Debiasing** (Zhang 2018) | Discriminator tries to predict `A` from the predictor's output; encoder fights it |
| **Lagrangian / constrained optimization** | Add `λ · fairness_violation(model)` to loss; tune `λ` |
| **Reductions** (Agarwal 2018, in Fairlearn) | Cast fair classification as a sequence of weighted ERM problems; provably optimal |
| **Fair-PCA / Counterfactual training** | Constrain hidden representations |
| **Group DRO** (Sagawa 2020) | Minimize worst-group loss; robust to spurious correlation |

**Reductions** (the Fairlearn `ExponentiatedGradient` / `GridSearch` reducers) are the most-used in 2026 because they (a) work with **any** classifier (sklearn API), (b) come with theoretical guarantees, (c) sweep the accuracy/fairness Pareto frontier.

```python
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
from sklearn.ensemble import GradientBoostingClassifier
constraint = EqualizedOdds(difference_bound=0.05)
mitigator = ExponentiatedGradient(GradientBoostingClassifier(), constraint)
mitigator.fit(X_train, y_train, sensitive_features=A_train)
```

**Group DRO** is the M93 / M98 callback — minimizing the worst-group loss prevents the model from achieving good average performance by sacrificing a small minority.


## 6 · Post-processing — fix predictions

Train the model normally, then transform `(score, A)` → `Ŷ` so that the *output* satisfies the chosen metric. The standard fix:

**Hardt-Price-Srebro 2016 (Equalized Odds post-processing).** For each group, find a per-group threshold (or randomized mixture of thresholds) such that the chosen fairness metric is satisfied. Closed-form: solve an LP.

| Pros | Cons |
|---|---|
| No retraining; works on any black-box model | Doesn't change the model; may sacrifice more accuracy than in-processing |
| Provably optimal for the post-hoc rule | Per-group thresholds = need `A` at inference (legal issue in some jurisdictions) |
| Trivial to deploy | Doesn't help if the underlying model is fundamentally biased |

**Calibration post-hoc.** Run M98's temperature scaling **per group** to fix calibration parity. Five lines per group.

> 🛠️ **Defensible-engineering pattern.** Most production teams ship **pre-processing (reweighing on labeled data)** + **in-processing (Group DRO or Reductions)** + **post-processing (Hardt fix or per-group calibration)**. Layered, like security; each catches what the others miss.


## 7 · The explainability canon

EU AI Act, Brazil LGPD, US ECOA give affected individuals the right to an explanation. The standard methods (PyTorch users go straight to **Captum**, the Meta unification library):

| Method | Granularity | Cost | Best for |
|---|---|---|---|
| **LIME** (Ribeiro 2016) | Local surrogate (linear) | ~thousands of perturbations per explanation | Tabular, text, image |
| **SHAP** (Lundberg 2017) | Shapley-value game theory | TreeSHAP exact for tree models; KernelSHAP ~slow | Tabular (industry default) |
| **Integrated Gradients** (Sundararajan 2017) | Path integral of gradients from baseline to input | ~50 forward+backward passes | Differentiable models (NN, Transformer) |
| **GradCAM / GradCAM++** (Selvaraju 2017) | Class-discriminative heatmap from CNN feature maps | 1 backward pass | CNN image classifiers |
| **Saliency / SmoothGrad** | Gradient × input or smoothed gradient | 1 backward (saliency); ~50 (SmoothGrad) | Quick visual proxy |
| **DeepLIFT** (Shrikumar 2017) | Reference-baseline activation differences | 1 forward+backward | Classification |
| **Attention attribution** (Abnar 2020) | Roll-out / attention flow through Transformer layers | 1 forward | LLM, ViT |
| **TCAV** (Kim 2018) | Concept activation vectors | Need labeled concept examples | Human-interpretable concept testing |
| **Influence functions** (Koh 2017) | Per-training-example influence on a test prediction | Expensive | Training-data attribution |

**Captum (PyTorch).** A single API exposing IG, DeepLIFT, GradCAM, LIME, Shapley, attention attribution, layer-conductance, neuron-conductance. **The 2026 default** for explainability work in PyTorch.

```python
from captum.attr import IntegratedGradients
ig = IntegratedGradients(model)
attr = ig.attribute(x, target=label, baselines=torch.zeros_like(x), n_steps=50)
```

> 🎯 **What "SHAP" actually is.** Shapley values from cooperative game theory: how much does each feature contribute to the prediction *relative to its average*, averaged over all feature-coalition orderings. **The only attribution method satisfying local accuracy + missingness + consistency simultaneously** (Lundberg 2017 theorem). TreeSHAP gives this exactly for tree ensembles in polynomial time — why SHAP became the tabular default.


## 8 · Counterfactual + anchor + contrastive explanations

The above methods say *which features mattered*. **Counterfactual explanations** answer the user's actual question: *what would have to change for the decision to flip?*

**Counterfactual (Wachter 2017):**
```
argmin_{x'} d(x, x')  subject to  f(x') ≠ f(x),  x' ∈ valid
```

"You were denied. If your income had been $5k higher and your credit utilization 10% lower, you would have been approved." **The EU AI Act's recital 71 explicitly mentions this style of explanation.**

| Method | Idea |
|---|---|
| **DiCE** (Mothilal 2020) | Diverse, plausible counterfactuals; gradient + genetic search |
| **CFProto** (Van Looveren 2021) | Prototype-guided; faster and more realistic |
| **GeCo / MACE / FACE** | SAT/SMT solvers for hard-constraint feasibility |
| **Wachter** (2017) | Vanilla L2 distance + classifier-loss optimization |

**Anchor explanations** (Ribeiro 2018) — find the *minimal feature subset* such that fixing them fully determines the prediction with high probability: *"If `income > $50k` and `credit_score > 700`, the model approves 99% of the time, regardless of other features."*

**Contrastive explanations** (Dhurandhar 2018) — Pertinent Positives (PP: minimal subset whose presence ensures the prediction) and Pertinent Negatives (PN: minimal feature whose absence would flip the prediction). PN is what users feel.

> 💬 **Why counterfactuals are taking over.** Compared to "feature `f` was important," users find "if `f` had been `v`, the result would have changed" *actionable*. Counterfactuals are how regulators want explanations delivered to affected persons.


## 9 · LLM-specific responsibility

Classical fairness methods predate LLMs. The 2024-2026 stack that's emerged for generative AI:

| Area | Tool / method | Where covered |
|---|---|---|
| **Refusal training** | Constitutional AI · RLAIF | **M89** |
| **Red-teaming evals** | AdvBench · XSTest · HarmBench · JailbreakBench | **M89, M91** |
| **Image / VLM evals** | Caption hallucination · typographic attacks | **M95** |
| **Demographic fairness** | BOLD · HolisticBias · StereoSet · CrowS-Pairs | This module |
| **Attention attribution** | Captum + transformer rollout | This module |
| **Mechanistic interp.** | Anthropic / OpenAI / DeepMind circuits research | M81 capstone preview |
| **Watermarking** | California SB-942 + Google SynthID + Stable Signature | M89 |
| **Toxicity classifiers** | Llama-Guard-3 · Perspective API · Detoxify | M89 |
| **Counterfactual fairness for LLMs** | Causal-debiasing of pretraining mix, instruction-tuning on counter-stereotypical examples | research-grade |

**Demographic-fairness LLM benchmarks:** BBQ (bias-benchmark for QA), BOLD (open-domain generation), HolisticBias (descriptor-prompt completions), StereoSet (stereotypical vs anti-stereotypical association). Run all four and report the gap; this is what frontier-lab system cards now do.

> 🔭 **2026 frontier-trend: causal mechanistic interpretability.** Anthropic's circuits papers, OpenAI's Sparse Autoencoders, DeepMind's TransformerLens — all locate specific *internal computations* responsible for biased / harmful behavior so it can be edited at the weights level rather than papered over at the output. Production maturity is 18-24 months out for safety-critical deployments.


## 10 · The 2026 production stack + 100-module epilogue

**Responsible-ML production stack (May 2026):**

| Layer | Tools | What it does |
|---|---|---|
| **Bias metrics** | Fairlearn · AIF360 · `aequitas` · `fairness-indicators` | Compute DP / EOpp / EO / DI; sweep Pareto frontier |
| **Mitigation** | Fairlearn Reductions · ExponentiatedGradient · GroupDRO | In-processing fair training |
| **Explainability** | SHAP · LIME · Captum · alibi · ELI5 | Per-prediction explanations |
| **Counterfactuals** | DiCE · alibi-explain · CFProto | What-if explanations |
| **Calibration / uncertainty** | torchuq · laplace-torch · mapie (M98) | Calibrated outputs + conformal sets |
| **LLM safety** | Llama-Guard-3 · Constitutional AI · Skywork-Reward · adversarial evals (M89) | Production guardrails |
| **Governance** | Model cards · system cards · datasheets · AI risk registers (NIST AI RMF) | Documentation + audit trail |
| **Monitoring** | EvidentlyAI · Arize · WhyLabs · Fiddler | Drift, fairness, and explanation monitoring |

**A 2026 production ML pipeline now has these layers minimum**: training (M99) + uncertainty (M98) + bias-audit (this module) + explainability (this module) + monitoring + model card. Skipping any of them creates regulatory exposure under EU AI Act / Colorado SB-205 / NYC AEDT for "consequential" decisions.

---

## 🎓 The 100-module epilogue

You started at **M1 — Python basics** and finished at **M100 — Fairness, Bias Mitigation, Explainability**. The course architecture:

| Phase | Modules | What you built |
|---|---|---|
| **1. Python + Data Foundations** | M1-M17 | Pandas, NumPy, viz, dashboards, ETL, EDA |
| **2. Modern ML Foundations** | M18-M27 | 6 model families, Transformers from scratch, diffusion, time-series, math + eval |
| **3-7. LLM Ecosystem** | M28-M54 | RAG, LangChain, LangGraph, fine-tuning, vector DBs, MLOps, Go / Rust / TS |
| **8. LLMs from Scratch** | M55-M63 | Tokenization → GPT-2 → SFT → KV-cache → Llama 3 → DPO from scratch |
| **9. Production Gaps** | M64-M73 | MCP, multimodal, distributed, RLHF, orchestration, agents, Spark |
| **10. Master-Plan Gaps** | M74-M81 | Security, A2A, classical RL, GPU networks, bare metal, FinOps, capstone |
| **11. Foundations Backfill** | M82-M85 | Linux + Bash, CNNs, RNN/LSTM/GRU, GAN/AE/VAE |
| **12. Production Extensions** | M86-M90 | Diffusion deep-dive, GraphRAG, synthetic data, CAI/RLAIF, edge AI |
| **13. Vision Robustness** | M91-M95 | Adversarial ML, Style Transfer + AdaIN, Training Tricks, Image Translation, Captioning + NLP-adv |
| **14. Theory & Responsibility** | **M96-M100** | GNNs, Normalizing Flows, Generalization + Bayesian, Optimization, **Fairness + Explainability** |

**100 notebooks. 100 docx files. ~8 MB of code. ~28 MB of explanations. One repository.**

The course covers — at production-relevant depth — every major topic shipped in modern ML / AI / data science as of May 2026: classical ML, deep learning, transformers, diffusion, LLMs, agents, RAG, GraphRAG, multimodal, edge AI, alignment, generalization theory, optimization, uncertainty, fairness. Nothing is left at "wave-of-the-hand."

> 🏁 **The closing thought.** The reason a curriculum at this size doesn't decay quickly is that it carries **lineages** — every M86-M100 module ends with a callback to M19 (attention), M86 (diffusion), M85 (VAE), or M89 (alignment), so the reader sees the *substrate* that doesn't change as well as the *snapshot* that does. Modules 1-85 are the substrate; modules 86-100 are the 2026 snapshot. **In 2027 the snapshots will need refreshing; the substrate won't.** Knowing the difference is the maturity threshold of senior ML engineering.

> 🎓 **Course complete. M1 → M100. Ship.**

## ✅ Recap

- **Legal frame**: EU AI Act, Colorado SB-205, NYC AEDT, FDA SaMD, Brazil LGPD all require bias evaluation + explainability for consequential decisions.
- **Fairness metric zoo**: DP, EOpp, EO, predictive parity, calibration parity, counterfactual fairness, individual fairness, 80% rule. **Choose one by use case.**
- **Impossibility theorem** (Chouldechova, Kleinberg): calibration + equal FPR + equal FNR ⇒ perfect model OR equal base rates. **No universal fair model exists.**
- **Mitigation**: pre-processing (reweighing, LFR) + in-processing (adversarial debiasing, Reductions, Group DRO) + post-processing (Hardt 2016 equalized-odds fix).
- **Explainability canon**: SHAP (tabular default), LIME, Integrated Gradients (NN), GradCAM (CNN), attention attribution (Transformer), DeepLIFT, TCAV, influence functions. **Captum unifies them in PyTorch.**
- **Counterfactual + anchor + contrastive** explanations are the *actionable* end-user explanation style — EU AI Act recital 71.
- **LLM responsibility**: Constitutional AI (M89) + Llama-Guard-3 (M89) + BBQ/BOLD/HolisticBias/StereoSet + mechanistic interpretability.
- **2026 production stack**: Fairlearn + AIF360 + Captum + SHAP + DiCE + EvidentlyAI + model cards + NIST AI RMF.

🎓 **Course complete. M1 → M100. Ship.**
